# Prueba de exploración — movilidad

Este notebook trabaja con las tres tablas de movilidad descritas en el reto:

- **M1** `movilidad entre barrios`
- **M2** `movilidad entre municipios`
- **M3** `movilidad entre barrios y municipios`

Los CSV originales deben estar en `data/raw/`. El notebook no modifica esos ficheros.

## 1. Preparación

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {ROOT.resolve()}")
print(f"Entrada:  {RAW.resolve()}")


In [ ]:
def find_csv(stem):
    """Busca un CSV por el nombre esperado, permitiendo sufijos y mayúsculas."""
    candidates = sorted(RAW.glob("*.csv"))
    exact = [p for p in candidates if p.stem.lower() == stem.lower()]
    matches = exact or [p for p in candidates if stem.lower() in p.stem.lower()]
    if not matches:
        raise FileNotFoundError(
            f"No se encontró un CSV para '{stem}' en {RAW}. "
            "Copia primero los datos originales en data/raw/."
        )
    if len(matches) > 1:
        print("Aviso: se encontraron varias coincidencias; se usa", matches[0].name)
    return matches[0]

def read_csv(stem):
    path = find_csv(stem)
    df = pd.read_csv(path, low_memory=False)
    df.columns = (
        pd.Index(df.columns).astype(str).str.strip().str.lower()
        .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
        .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    )
    print(f"{path.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    return df

def profile(df):
    return (pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_pct": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=True),
    }).sort_values(["nulos_pct", "unicos"], ascending=[False, True]))

def as_date(df, column="day"):
    if column in df:
        df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


## 2. Carga de M1, M2 y M3

Si los nombres reales contienen un prefijo o sufijo, la búsqueda por fragmento los encontrará.

In [ ]:
M1 = read_csv("movilidad_barrios")
M2 = read_csv("movilidad_municipios")
M3 = read_csv("movilidad_barrios_municipios")

for nombre, df in {"M1": M1, "M2": M2, "M3": M3}.items():
    print(f"\n--- {nombre} ---")
    display(profile(df))


## 3. Comprobaciones de calidad

In [ ]:
required_common = {"day", "day_of_the_week", "month", "origen", "viajes"}
for nombre, df in {"M1": M1, "M2": M2, "M3": M3}.items():
    missing = required_common - set(df.columns)
    assert not missing, f"{nombre}: faltan columnas {sorted(missing)}"
    df = as_date(df)
    print(nombre, "rango:", df["day"].min(), "→", df["day"].max())
    print("segmentos:", sorted(df["origen"].dropna().astype(str).unique()))
    print("viajes negativos:", int((pd.to_numeric(df["viajes"], errors="coerce") < 0).sum()))


## 4. Evolución diaria y mensual

`viajes` mide desplazamientos, no personas únicas. Por eso se puede sumar entre flujos, pero no interpretarlo como visitantes.

In [ ]:
M1 = as_date(M1)
M1["viajes"] = pd.to_numeric(M1["viajes"], errors="coerce")
serie = (M1.dropna(subset=["day"]).groupby("day", as_index=False)["viajes"].sum())
serie.plot(x="day", y="viajes", figsize=(13, 4), title="M1 — viajes diarios entre barrios")
plt.ylabel("Viajes")
plt.xlabel("Día")
plt.show()

mensual = (M1.assign(periodo=M1["day"].dt.to_period("M").astype(str))
             .groupby("periodo", as_index=False)["viajes"].sum())
display(mensual.tail(12))


## 5. Flujos principales

La siguiente celda funciona cuando el fichero contiene los nombres de origen y destino. Si solo tiene códigos, se puede sustituir la lista de columnas por esos códigos.

In [ ]:
def top_flows(df, n=15):
    origin = next((c for c in ["barrio_origen_name", "municipio_origen_name", "barrio_origen", "municipio_origen"] if c in df), None)
    destination = next((c for c in ["barrio_destino_name", "municipio_destino_name", "barrio_destino", "municipio_destino"] if c in df), None)
    if not origin or not destination:
        raise KeyError("No se encontraron columnas de origen/destino")
    return (df.groupby([origin, destination], dropna=False, as_index=False)["viajes"]
              .sum().sort_values("viajes", ascending=False).head(n))

display(top_flows(M1))


## 6. Comparación por segmento de visitante

In [ ]:
segmentos = (M1.groupby("origen", dropna=False, as_index=False)["viajes"]
              .sum().sort_values("viajes", ascending=False))
segmentos.plot.bar(x="origen", y="viajes", legend=False, figsize=(9, 4), title="M1 — viajes por origen")
plt.ylabel("Viajes")
plt.xlabel("Origen")
plt.xticks(rotation=0)
plt.show()
display(segmentos)
